# Liane's Library – MySQL Test Notebook

Dieses Notebook erstellt und testet eine einfache Bibliotheksdatenbank mit drei Kerntabellen:

- `books`
- `borrowers`
- `loans`

Zusätzlich werden zwei Views erstellt:

- `v_loan_overview`
- `v_open_loans`

Das Notebook ist als einfache Lern- und Testumgebung gedacht. Es enthält Schema-Erstellung, Beispieldaten, CRUD-Befehle, Joins und Integritätstests.

## 1. Vorbereitung

Benötigt werden:

- MySQL 8.0 oder neuer
- Python 3
- Jupyter Notebook oder JupyterLab
- `mysql-connector-python`
- `pandas`

Falls das MySQL-Paket fehlt, führe diese Zelle einmal aus:

```python
%pip install mysql-connector-python pandas
```

Danach den Kernel neu starten und das Notebook erneut von oben ausführen.

In [48]:
# Abhängigkeiten prüfen
import os
from datetime import date, timedelta

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    pd = None
    PANDAS_AVAILABLE = False

try:
    import mysql.connector
    from mysql.connector import Error, IntegrityError
    MYSQL_CONNECTOR_AVAILABLE = True
except ImportError:
    mysql = None
    Error = Exception
    IntegrityError = Exception
    MYSQL_CONNECTOR_AVAILABLE = False

print(f"Pandas verfügbar: {PANDAS_AVAILABLE}")
print(f"MySQL Connector verfügbar: {MYSQL_CONNECTOR_AVAILABLE}")

if not MYSQL_CONNECTOR_AVAILABLE:
    print("\nInstallation ausführen:")
    print("%pip install mysql-connector-python pandas")

Pandas verfügbar: True
MySQL Connector verfügbar: True


In [49]:
%pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Verbindung konfigurieren

Trage deine lokalen MySQL-Zugangsdaten ein.

`EXECUTE_DATABASE_COMMANDS` bleibt zunächst auf `False`. Setze den Wert erst auf `True`, wenn:

1. MySQL läuft,
2. der Connector installiert ist,
3. Benutzername und Passwort korrekt sind.

In [ ]:
# Sicherheits-Schalter
EXECUTE_DATABASE_COMMANDS = True

DB_NAME = "lianes_library"

DB_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
}

safe_config = {
    **DB_CONFIG,
    "password": "***" if DB_CONFIG["password"] else ""
}

print("Konfiguration:", safe_config)
print("Datenbank:", DB_NAME)
print("Befehle ausführen:", EXECUTE_DATABASE_COMMANDS)

Konfiguration: {'host': 'localhost', 'port': 3306, 'user': 'root', 'password': ''}
Datenbank: lianes_library
Befehle ausführen: False


In [69]:
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    print("Verbindung OK")
    conn.close()
except mysql.connector.Error as err:
    print("Verbindungsfehler:")
    print(err)

Verbindung OK


## 3. Hilfsfunktionen

Die Funktionen kapseln Verbindungen, Änderungen und Abfragen. Dadurch bleiben die späteren Testzellen übersichtlich.

In [51]:
def require_database_execution():
    if not EXECUTE_DATABASE_COMMANDS:
        raise RuntimeError(
            "EXECUTE_DATABASE_COMMANDS ist False. "
            "Zugangsdaten prüfen und den Schalter anschließend auf True setzen."
        )
    if not MYSQL_CONNECTOR_AVAILABLE:
        raise RuntimeError(
            "mysql-connector-python fehlt. "
            "Installiere das Paket mit %pip install mysql-connector-python."
        )


def get_connection(database=None):
    require_database_execution()

    config = DB_CONFIG.copy()
    if database is not None:
        config["database"] = database

    return mysql.connector.connect(**config)


def execute_statement(sql, params=None, database=DB_NAME):
    connection = get_connection(database)
    cursor = connection.cursor()

    try:
        cursor.execute(sql, params or ())
        connection.commit()
        return cursor.rowcount
    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


def execute_many(sql, parameter_rows, database=DB_NAME):
    connection = get_connection(database)
    cursor = connection.cursor()

    try:
        cursor.executemany(sql, parameter_rows)
        connection.commit()
        return cursor.rowcount
    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


def query_dataframe(sql, params=None, database=DB_NAME):
    connection = get_connection(database)

    try:
        if PANDAS_AVAILABLE:
            return pd.read_sql(sql, connection, params=params)

        cursor = connection.cursor(dictionary=True)
        cursor.execute(sql, params or ())
        rows = cursor.fetchall()
        cursor.close()
        return rows
    finally:
        connection.close()


print("Hilfsfunktionen geladen.")

Hilfsfunktionen geladen.


## 4. Datenbankschema erstellen

### Beziehungen

```text
books      1 ───── n loans n ───── 1 borrowers
```

- Ein Buch kann über die Zeit mehrfach verliehen werden.
- Eine Person kann mehrere Bücher ausleihen.
- `loans` speichert die komplette Historie.
- Ein Unique-Constraint verhindert zwei gleichzeitig offene Ausleihen desselben Buches.

In [52]:
schema_statements = [
    f"""
    CREATE DATABASE IF NOT EXISTS {DB_NAME}
        CHARACTER SET utf8mb4
        COLLATE utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS books (
        book_id INT UNSIGNED NOT NULL AUTO_INCREMENT,
        title VARCHAR(255) NOT NULL,
        author VARCHAR(255) NOT NULL,
        isbn VARCHAR(20) NULL,
        category VARCHAR(100) NULL,
        shelf_location VARCHAR(100) NULL,
        notes TEXT NULL,
        is_active BOOLEAN NOT NULL DEFAULT TRUE,
        created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
        updated_at DATETIME NOT NULL
            DEFAULT CURRENT_TIMESTAMP
            ON UPDATE CURRENT_TIMESTAMP,

        CONSTRAINT pk_books PRIMARY KEY (book_id),
        CONSTRAINT uq_books_isbn UNIQUE (isbn),
        CONSTRAINT chk_books_title
            CHECK (CHAR_LENGTH(TRIM(title)) > 0),
        CONSTRAINT chk_books_author
            CHECK (CHAR_LENGTH(TRIM(author)) > 0),

        INDEX idx_books_title (title),
        INDEX idx_books_author (author),
        INDEX idx_books_category (category),
        INDEX idx_books_is_active (is_active)
    ) ENGINE = InnoDB
    """,
    """
    CREATE TABLE IF NOT EXISTS borrowers (
        borrower_id INT UNSIGNED NOT NULL AUTO_INCREMENT,
        name VARCHAR(150) NOT NULL,
        email VARCHAR(254) NULL,
        phone VARCHAR(30) NULL,
        relationship VARCHAR(50) NULL,
        notes TEXT NULL,
        is_active BOOLEAN NOT NULL DEFAULT TRUE,
        created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
        updated_at DATETIME NOT NULL
            DEFAULT CURRENT_TIMESTAMP
            ON UPDATE CURRENT_TIMESTAMP,

        CONSTRAINT pk_borrowers PRIMARY KEY (borrower_id),
        CONSTRAINT uq_borrowers_email UNIQUE (email),
        CONSTRAINT chk_borrowers_name
            CHECK (CHAR_LENGTH(TRIM(name)) > 0),

        INDEX idx_borrowers_name (name),
        INDEX idx_borrowers_is_active (is_active)
    ) ENGINE = InnoDB
    """,
    """
    CREATE TABLE IF NOT EXISTS loans (
        loan_id INT UNSIGNED NOT NULL AUTO_INCREMENT,
        book_id INT UNSIGNED NOT NULL,
        borrower_id INT UNSIGNED NOT NULL,
        loan_date DATE NOT NULL DEFAULT (CURRENT_DATE),
        due_date DATE NULL,
        return_date DATE NULL,
        notes TEXT NULL,
        created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
        updated_at DATETIME NOT NULL
            DEFAULT CURRENT_TIMESTAMP
            ON UPDATE CURRENT_TIMESTAMP,

        open_loan_book_id INT UNSIGNED
            GENERATED ALWAYS AS (
                CASE
                    WHEN return_date IS NULL THEN book_id
                    ELSE NULL
                END
            ) STORED,

        CONSTRAINT pk_loans PRIMARY KEY (loan_id),

        CONSTRAINT fk_loans_book
            FOREIGN KEY (book_id)
            REFERENCES books (book_id)
            ON UPDATE CASCADE
            ON DELETE RESTRICT,

        CONSTRAINT fk_loans_borrower
            FOREIGN KEY (borrower_id)
            REFERENCES borrowers (borrower_id)
            ON UPDATE CASCADE
            ON DELETE RESTRICT,

        CONSTRAINT chk_loans_due_date
            CHECK (due_date IS NULL OR due_date >= loan_date),

        CONSTRAINT chk_loans_return_date
            CHECK (return_date IS NULL OR return_date >= loan_date),

        CONSTRAINT uq_loans_one_open_loan_per_book
            UNIQUE (open_loan_book_id),

        INDEX idx_loans_book_id (book_id),
        INDEX idx_loans_borrower_id (borrower_id),
        INDEX idx_loans_loan_date (loan_date),
        INDEX idx_loans_due_date (due_date),
        INDEX idx_loans_return_date (return_date)
    ) ENGINE = InnoDB
    """,
    """
    CREATE OR REPLACE VIEW v_loan_overview AS
    SELECT
        l.loan_id,
        b.book_id,
        b.title,
        b.author,
        b.isbn,
        b.category,
        b.shelf_location,
        br.borrower_id,
        br.name AS borrower_name,
        br.email AS borrower_email,
        br.phone AS borrower_phone,
        br.relationship,
        l.loan_date,
        l.due_date,
        l.return_date,
        l.notes AS loan_notes,
        CASE
            WHEN l.return_date IS NOT NULL THEN 'Returned'
            WHEN l.due_date IS NOT NULL
                 AND l.due_date < CURRENT_DATE THEN 'Overdue'
            ELSE 'Loaned'
        END AS loan_status
    FROM loans AS l
    INNER JOIN books AS b
        ON l.book_id = b.book_id
    INNER JOIN borrowers AS br
        ON l.borrower_id = br.borrower_id
    """,
    """
    CREATE OR REPLACE VIEW v_open_loans AS
    SELECT
        loan_id,
        book_id,
        title,
        author,
        borrower_id,
        borrower_name,
        borrower_email,
        borrower_phone,
        loan_date,
        due_date,
        loan_status
    FROM v_loan_overview
    WHERE return_date IS NULL
    """
]

print(f"{len(schema_statements)} Schema-Befehle vorbereitet.")

6 Schema-Befehle vorbereitet.


In [53]:
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    print("Verbindung OK")
    conn.close()
except mysql.connector.Error as err:
    print("Verbindungsfehler:")
    print(err)

Verbindungsfehler:
1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)


In [54]:
# Datenbank und Tabellen erstellen
if EXECUTE_DATABASE_COMMANDS:
    try:
        # CREATE DATABASE wird ohne ausgewählte Datenbank ausgeführt
        execute_statement(schema_statements[0], database=None)

        # Tabellen und Views werden in lianes_library erstellt
        for statement in schema_statements[1:]:
            execute_statement(statement)

        print("Datenbank, Tabellen und Views wurden erstellt.")
    except mysql.connector.Error as error:
        print("MySQL-Verbindung fehlgeschlagen. Bitte überprüfe die Zugangsdaten in der Konfiguration:")
        print(error)
else:
    print("Übersprungen: EXECUTE_DATABASE_COMMANDS ist False.")

Übersprungen: EXECUTE_DATABASE_COMMANDS ist False.


## 5. Schema kontrollieren

Diese Tests zeigen die erstellten Tabellen, Views und Spalten.

In [55]:
if EXECUTE_DATABASE_COMMANDS:
    display(query_dataframe("SHOW FULL TABLES"))
    display(query_dataframe("DESCRIBE books"))
    display(query_dataframe("DESCRIBE borrowers"))
    display(query_dataframe("DESCRIBE loans"))
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 6. Beispieldaten einfügen

Die Inserts sind weitgehend wiederholbar:

- Bücher werden anhand der ISBN erkannt.
- Personen werden anhand der E-Mail-Adresse erkannt.
- Vorhandene Daten werden aktualisiert statt dupliziert.

In [56]:
sample_books = [
    (
        "The Hobbit",
        "J. R. R. Tolkien",
        "9780261102217",
        "Fantasy",
        "Shelf 2",
        "Favourite edition"
    ),
    (
        "1984",
        "George Orwell",
        "9780451524935",
        "Dystopian Fiction",
        "Shelf 1",
        None
    ),
    (
        "Dune",
        "Frank Herbert",
        "9780441172719",
        "Science Fiction",
        "Shelf 3",
        None
    )
]

sample_borrowers = [
    (
        "Anna Schmidt",
        "anna@example.com",
        "0151 12345678",
        "Colleague",
        None
    ),
    (
        "Max Müller",
        "max@example.com",
        "0170 12345678",
        "Friend",
        "Usually returns books on time"
    )
]

insert_book_sql = """
INSERT INTO books (
    title,
    author,
    isbn,
    category,
    shelf_location,
    notes
)
VALUES (%s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    title = VALUES(title),
    author = VALUES(author),
    category = VALUES(category),
    shelf_location = VALUES(shelf_location),
    notes = VALUES(notes),
    is_active = TRUE
"""

insert_borrower_sql = """
INSERT INTO borrowers (
    name,
    email,
    phone,
    relationship,
    notes
)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    name = VALUES(name),
    phone = VALUES(phone),
    relationship = VALUES(relationship),
    notes = VALUES(notes),
    is_active = TRUE
"""

if EXECUTE_DATABASE_COMMANDS:
    execute_many(insert_book_sql, sample_books)
    execute_many(insert_borrower_sql, sample_borrowers)
    print("Beispieldaten eingefügt oder aktualisiert.")
else:
    print("Beispieldaten vorbereitet, aber nicht eingefügt.")

Beispieldaten vorbereitet, aber nicht eingefügt.


## 7. Basisabfragen testen

In [57]:
if EXECUTE_DATABASE_COMMANDS:
    print("Bücher")
    display(query_dataframe(
        """
        SELECT
            book_id,
            title,
            author,
            isbn,
            category,
            shelf_location,
            is_active
        FROM books
        ORDER BY title
        """
    ))

    print("Personen")
    display(query_dataframe(
        """
        SELECT
            borrower_id,
            name,
            email,
            phone,
            relationship,
            is_active
        FROM borrowers
        ORDER BY name
        """
    ))
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 8. Erste Ausleihe anlegen

Das Beispiel verleiht `The Hobbit` an `Anna Schmidt`.

Die IDs werden über stabile Werte wie ISBN und E-Mail-Adresse ermittelt. Dadurch müssen keine festen IDs angenommen werden.

In [58]:
if EXECUTE_DATABASE_COMMANDS:
    book_result = query_dataframe(
        "SELECT book_id FROM books WHERE isbn = %s",
        params=("9780261102217",)
    )
    borrower_result = query_dataframe(
        "SELECT borrower_id FROM borrowers WHERE email = %s",
        params=("anna@example.com",)
    )

    book_id = int(book_result.iloc[0]["book_id"])
    borrower_id = int(borrower_result.iloc[0]["borrower_id"])

    loan_date = date.today()
    due_date = loan_date + timedelta(days=14)

    open_loan = query_dataframe(
        """
        SELECT loan_id
        FROM loans
        WHERE book_id = %s
          AND return_date IS NULL
        """,
        params=(book_id,)
    )

    if open_loan.empty:
        execute_statement(
            """
            INSERT INTO loans (
                book_id,
                borrower_id,
                loan_date,
                due_date,
                notes
            )
            VALUES (%s, %s, %s, %s, %s)
            """,
            params=(
                book_id,
                borrower_id,
                loan_date,
                due_date,
                "Book handed over in good condition."
            )
        )
        print("Ausleihe wurde angelegt.")
    else:
        print("Das Buch besitzt bereits eine offene Ausleihe.")
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 9. Join und Views testen

Die View übernimmt die Joins zwischen `loans`, `books` und `borrowers`.

In [59]:
if EXECUTE_DATABASE_COMMANDS:
    display(query_dataframe(
        """
        SELECT *
        FROM v_loan_overview
        ORDER BY loan_date DESC, loan_id DESC
        """
    ))

    print("Nur offene Ausleihen")
    display(query_dataframe(
        """
        SELECT *
        FROM v_open_loans
        ORDER BY due_date
        """
    ))
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 10. Verfügbare Bücher anzeigen

Ein Buch gilt als verfügbar, wenn:

- es aktiv ist und
- keine offene Ausleihe besitzt.

In [60]:
available_books_sql = """
SELECT
    b.book_id,
    b.title,
    b.author,
    b.category,
    b.shelf_location
FROM books AS b
LEFT JOIN loans AS l
    ON b.book_id = l.book_id
   AND l.return_date IS NULL
WHERE b.is_active = TRUE
  AND l.loan_id IS NULL
ORDER BY b.title
"""

if EXECUTE_DATABASE_COMMANDS:
    display(query_dataframe(available_books_sql))
else:
    print(available_books_sql)


SELECT
    b.book_id,
    b.title,
    b.author,
    b.category,
    b.shelf_location
FROM books AS b
LEFT JOIN loans AS l
    ON b.book_id = l.book_id
   AND l.return_date IS NULL
WHERE b.is_active = TRUE
  AND l.loan_id IS NULL
ORDER BY b.title



## 11. Integritätstest: doppelte offene Ausleihe verhindern

Der folgende Test versucht absichtlich, dasselbe Buch ein zweites Mal auszuleihen.

Erwartetes Ergebnis: MySQL lehnt den Datensatz wegen des Unique-Constraints ab.

In [61]:
if EXECUTE_DATABASE_COMMANDS:
    try:
        book_id = int(
            query_dataframe(
                "SELECT book_id FROM books WHERE isbn = %s",
                params=("9780261102217",)
            ).iloc[0]["book_id"]
        )

        borrower_id = int(
            query_dataframe(
                "SELECT borrower_id FROM borrowers WHERE email = %s",
                params=("max@example.com",)
            ).iloc[0]["borrower_id"]
        )

        execute_statement(
            """
            INSERT INTO loans (
                book_id,
                borrower_id,
                loan_date,
                due_date
            )
            VALUES (%s, %s, CURRENT_DATE, DATE_ADD(CURRENT_DATE, INTERVAL 7 DAY))
            """,
            params=(book_id, borrower_id)
        )

        print("Unerwartet: Die zweite offene Ausleihe wurde gespeichert.")

    except IntegrityError as error:
        print("Erwarteter Fehler: Das Buch ist bereits ausgeliehen.")
        print(error)
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 12. Buch als zurückgegeben markieren

Die Ausleihe wird nicht gelöscht. Stattdessen wird `return_date` gesetzt, damit die Historie erhalten bleibt.

In [62]:
if EXECUTE_DATABASE_COMMANDS:
    open_hobbit_loan = query_dataframe(
        """
        SELECT l.loan_id
        FROM loans AS l
        INNER JOIN books AS b
            ON l.book_id = b.book_id
        WHERE b.isbn = %s
          AND l.return_date IS NULL
        ORDER BY l.loan_id DESC
        LIMIT 1
        """,
        params=("9780261102217",)
    )

    if open_hobbit_loan.empty:
        print("Keine offene Ausleihe gefunden.")
    else:
        loan_id = int(open_hobbit_loan.iloc[0]["loan_id"])

        execute_statement(
            """
            UPDATE loans
            SET return_date = CURRENT_DATE
            WHERE loan_id = %s
              AND return_date IS NULL
            """,
            params=(loan_id,)
        )

        print(f"Ausleihe {loan_id} wurde als zurückgegeben markiert.")

    display(query_dataframe(
        """
        SELECT *
        FROM v_loan_overview
        ORDER BY loan_id DESC
        """
    ))
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 13. Buch und Person aktualisieren

In [63]:
if EXECUTE_DATABASE_COMMANDS:
    execute_statement(
        """
        UPDATE books
        SET shelf_location = %s,
            notes = %s
        WHERE isbn = %s
        """,
        params=(
            "Living Room – Shelf 2",
            "Updated through notebook test",
            "9780261102217"
        )
    )

    execute_statement(
        """
        UPDATE borrowers
        SET phone = %s,
            relationship = %s
        WHERE email = %s
        """,
        params=(
            "0151 99999999",
            "Friend and colleague",
            "anna@example.com"
        )
    )

    print("Buch und Person wurden aktualisiert.")

    display(query_dataframe(
        """
        SELECT book_id, title, shelf_location, notes
        FROM books
        WHERE isbn = '9780261102217'
        """
    ))

    display(query_dataframe(
        """
        SELECT borrower_id, name, phone, relationship
        FROM borrowers
        WHERE email = 'anna@example.com'
        """
    ))
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 14. Soft Delete testen

Statt Datensätze mit Historie endgültig zu löschen, werden sie deaktiviert.

Das ist später für Streamlit sinnvoll:

```sql
UPDATE books
SET is_active = FALSE
WHERE book_id = ?;
```

In [64]:
if EXECUTE_DATABASE_COMMANDS:
    execute_statement(
        """
        UPDATE books
        SET is_active = FALSE
        WHERE isbn = %s
        """,
        params=("9780451524935",)
    )

    print("Das Buch '1984' wurde deaktiviert.")

    display(query_dataframe(
        """
        SELECT
            book_id,
            title,
            author,
            is_active
        FROM books
        ORDER BY title
        """
    ))

    # Testdaten wieder aktivieren, damit das Notebook wiederholbar bleibt
    execute_statement(
        """
        UPDATE books
        SET is_active = TRUE
        WHERE isbn = %s
        """,
        params=("9780451524935",)
    )

    print("Das Testbuch wurde anschließend wieder aktiviert.")
else:
    print("Übersprungen: Keine Datenbankverbindung aktiviert.")

Übersprungen: Keine Datenbankverbindung aktiviert.


## 15. Zusätzliche Testabfragen

In [65]:
test_queries = {
    "Anzahl Bücher": """
        SELECT COUNT(*) AS book_count
        FROM books
        WHERE is_active = TRUE
    """,
    "Anzahl Personen": """
        SELECT COUNT(*) AS borrower_count
        FROM borrowers
        WHERE is_active = TRUE
    """,
    "Ausleihen je Person": """
        SELECT
            br.borrower_id,
            br.name,
            COUNT(l.loan_id) AS total_loans,
            SUM(l.return_date IS NULL) AS open_loans
        FROM borrowers AS br
        LEFT JOIN loans AS l
            ON br.borrower_id = l.borrower_id
        GROUP BY
            br.borrower_id,
            br.name
        ORDER BY total_loans DESC, br.name
    """,
    "Am häufigsten verliehene Bücher": """
        SELECT
            b.book_id,
            b.title,
            b.author,
            COUNT(l.loan_id) AS loan_count
        FROM books AS b
        LEFT JOIN loans AS l
            ON b.book_id = l.book_id
        GROUP BY
            b.book_id,
            b.title,
            b.author
        ORDER BY loan_count DESC, b.title
    """,
    "Überfällige Ausleihen": """
        SELECT *
        FROM v_loan_overview
        WHERE loan_status = 'Overdue'
        ORDER BY due_date
    """
}

if EXECUTE_DATABASE_COMMANDS:
    for query_name, sql in test_queries.items():
        print(f"\n{query_name}")
        display(query_dataframe(sql))
else:
    for query_name, sql in test_queries.items():
        print(f"\n--- {query_name} ---")
        print(sql.strip())


--- Anzahl Bücher ---
SELECT COUNT(*) AS book_count
        FROM books
        WHERE is_active = TRUE

--- Anzahl Personen ---
SELECT COUNT(*) AS borrower_count
        FROM borrowers
        WHERE is_active = TRUE

--- Ausleihen je Person ---
SELECT
            br.borrower_id,
            br.name,
            COUNT(l.loan_id) AS total_loans,
            SUM(l.return_date IS NULL) AS open_loans
        FROM borrowers AS br
        LEFT JOIN loans AS l
            ON br.borrower_id = l.borrower_id
        GROUP BY
            br.borrower_id,
            br.name
        ORDER BY total_loans DESC, br.name

--- Am häufigsten verliehene Bücher ---
SELECT
            b.book_id,
            b.title,
            b.author,
            COUNT(l.loan_id) AS loan_count
        FROM books AS b
        LEFT JOIN loans AS l
            ON b.book_id = l.book_id
        GROUP BY
            b.book_id,
            b.title,
            b.author
        ORDER BY loan_count DESC, b.title

--- Überfällige A

## 16. Optionaler Reset

Diese Zelle löscht die komplette Testdatenbank. Sie ist absichtlich doppelt abgesichert.

Nur verwenden, wenn wirklich alle Tabellen und Daten entfernt werden sollen.

In [66]:
ALLOW_DATABASE_RESET = False

if EXECUTE_DATABASE_COMMANDS and ALLOW_DATABASE_RESET:
    execute_statement(
        f"DROP DATABASE IF EXISTS {DB_NAME}",
        database=None
    )
    print(f"Datenbank '{DB_NAME}' wurde gelöscht.")
else:
    print("Reset nicht ausgeführt.")
    print("Zum Löschen müssen beide Schalter True sein:")
    print("- EXECUTE_DATABASE_COMMANDS")
    print("- ALLOW_DATABASE_RESET")

Reset nicht ausgeführt.
Zum Löschen müssen beide Schalter True sein:
- EXECUTE_DATABASE_COMMANDS
- ALLOW_DATABASE_RESET


## Nächste sinnvolle Schritte

Nach erfolgreichem Test kann Streamlit dieselben Aktionen über Formulare ausführen:

1. Bücher anzeigen und ergänzen
2. Personen verwalten
3. Verfügbare Bücher auswählen
4. Ausleihe speichern
5. Offene Ausleihen anzeigen
6. Buch als zurückgegeben markieren

Die View `v_open_loans` eignet sich direkt für die Hauptübersicht der Streamlit-App.